# Introduction (TBD)

# Libraries and Global Variables

In [ ]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from pathlib import Path

In [ ]:
# Seed for ML Algorithms
SEED = 81

# Paths
PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()
)
GENERATED_REPORTS = PROJECT / "reports" / "generated"

DIABETES_CSV = PROJECT / "data" / "raw" / "diabetic_data.csv"
IDS_MAPING_CSV = PROJECT / "data" / "raw" / "IDS_mapping.csv"

## Diabetes main fact table

In [ ]:
# Read diabetes CSV
df_diabetes = pd.read_csv(DIABETES_CSV, delimiter=",", na_values="?")
df_diabetes.head()

## IDS Mapping Tables
TODO: Add small explanation behind logic of generator 

In [ ]:
def process_ids_tables():
    # Read IDS mapping CSV
    df_ids_map = pd.read_csv(IDS_MAPING_CSV, header=None, names=range(2), delimiter=",")

    # Define table names
    table_names = ["admission_type", "discharge_disposition_id", "admission_source_id"]

    groups = df_ids_map[0].isin(table_names).cumsum()

    tables = {g.iloc[0, 0]: g.iloc[0:-1] for k, g in df_ids_map.groupby(groups)}

    for name, table in tables.items():
        # Create dataframe
        df = pd.DataFrame(table)

        # Promote first line to header
        df.columns = df.iloc[0]
        df = df.iloc[1:].reset_index(drop=True)

        yield df


ids_mapping_dataframes = process_ids_tables()

# Use generator to create our dataframes for each table

# Admission Types
df_admission_type = next(ids_mapping_dataframes)

# Discharge Disposition
df_discharge_disposition = next(ids_mapping_dataframes)

# Admission Source
df_admission_source = next(ids_mapping_dataframes)

# EDA

Just for starters, we'll create a Profile Report for the diabetes main fact table

In [ ]:
profile = ProfileReport(df_diabetes, title="Diabetes Data Profile Report")
profile.config.html.inline = True
profile.to_file(GENERATED_REPORTS / "profile-report-diabests.html")

# Data Exploration and Preprocessing

## Assigned Variables

| Group | Variables |
|---|---|
| Identifiers | `encounter_id`, `patient_nbr` |
| Sociodemographic | `race`, `gender`, `age`, `weight` |
| Diagnoses | `diag_1`, `diag_2`, `diag_3`, `number_diagnoses` |
| Lab results | `max_glu_serum`, `A1Cresult` |
| Medication summary | `change`, `diabetesMed` |

In [ ]:
# Check for fully duplicated rows
df_diabetes.duplicated().sum()

### Identifiers

In [ ]:
# Check that each encounter_id appears only once
df_diabetes["encounter_id"].is_unique

In [ ]:
# Count unique patients
df_diabetes["patient_nbr"].nunique()

In [ ]:
# Count patients with more than one encounter
(df_diabetes["patient_nbr"].value_counts() > 1).sum()

**Identifiers decision:** `encounter_id` is unique per row and `patient_nbr` repeats across encounters. Neither is used as a predictor. `patient_nbr` is kept to split train/test by patient, so the same patient never appears in both sets (avoids leakage).